In [0]:
# load libraries
import pandas as pd
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split


### Data Preparation and Research Question Set Up

This starter notebook uses an F1 results-style dataset from the Databricks volume. Each row represents one driver in one race.

The starter research question is: **Can we predict whether a driver will finish on the podium?**

So for each driver-race row, we create a new binary variable:

- `podium_finish = 1` if the driver finished in position 1, 2, or 3
- `podium_finish = 0` otherwise

This is a simple classification target that works well for a first MLflow experiment.


In [0]:
# check files in the class volume and update DATASET_PATH below if needed
VOLUME_PATH = "/Volumes/gr5069/raw/f1_data"
display(dbutils.fs.ls(VOLUME_PATH))


In [0]:
# load one F1 dataset from the volume
DATASET_PATH = "/Volumes/gr5069/raw/f1_data/results.csv"
DATASET_FORMAT = "csv"  # change to "parquet" if needed

if DATASET_FORMAT == "csv":
    df_f1 = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(DATASET_PATH)
    )
elif DATASET_FORMAT == "parquet":
    df_f1 = spark.read.parquet(DATASET_PATH)
else:
    raise ValueError("DATASET_FORMAT must be 'csv' or 'parquet'")

display(df_f1)


In [0]:
# create a pandas copy for modeling
df = df_f1.toPandas().copy()
display(df.head())


#### Choose Predictor Variables

This baseline keeps a small set of simple predictors so you can get a working experiment first and improve it later.


In [0]:
# create the target column
if "positionOrder" not in df.columns:
    raise ValueError("This notebook expects a results-style dataset with a 'positionOrder' column.")

df["positionOrder"] = pd.to_numeric(df["positionOrder"], errors="coerce")
df["podium_finish"] = (df["positionOrder"] <= 3).astype("Int64")

display(df[["positionOrder", "podium_finish"]].head())


In [0]:
# select columns to keep for modeling
candidate_feature_columns = ["grid", "driverId", "constructorId", "raceId", "number"]
feature_cols = [column for column in candidate_feature_columns if column in df.columns]

if not feature_cols:
    raise ValueError("None of the baseline feature columns were found. Update candidate_feature_columns.")

model_df = df[feature_cols + ["podium_finish"]].copy()

for column in feature_cols:
    model_df[column] = pd.to_numeric(model_df[column], errors="coerce")

model_df = model_df.dropna(subset=["podium_finish"])
model_df = model_df.fillna(-1)
model_df["podium_finish"] = model_df["podium_finish"].astype(int)

display(model_df.head())


In [0]:
# double check missing values and class balance
display(model_df.isna().sum().reset_index(name="missing_count"))
display(model_df["podium_finish"].value_counts().rename_axis("podium_finish").reset_index(name="count"))


#### Build and Train the Model

To get started quickly, this notebook uses a standard stratified train/test split. Once you have a clean baseline, you can improve it with more features or a time-based split.


In [0]:
# split data into train and test sets
X = model_df[feature_cols]
y = model_df["podium_finish"].values.ravel()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Feature columns:", feature_cols)


#### Train Random Forest and Log with MLflow

First, run one simple MLflow experiment to create a tracked baseline run.


In [0]:
# basic MLflow run
with mlflow.start_run(run_name="Basic RF Experiment") as run:
    rf = RandomForestClassifier(random_state=42)
    rf.fit(X_train, y_train)
    predictions = rf.predict(X_test)

    mlflow.sklearn.log_model(rf, "random-forest-model")

    accuracy = accuracy_score(y_test, predictions)
    f1 = f1_score(y_test, predictions, zero_division=0)

    print("accuracy:", accuracy)
    print("f1:", f1)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1", f1)

    runID = run.info.run_id
    experimentID = run.info.experiment_id

    print("Inside MLflow Run with run_id {} and experiment_id {}".format(runID, experimentID))


In [0]:
def log_rf(experimentID, run_name, params, X_train, X_test, y_train, y_test, feature_cols):
    import tempfile
    from pathlib import Path
    from sklearn.metrics import (
        ConfusionMatrixDisplay,
        balanced_accuracy_score,
        classification_report,
        roc_auc_score,
    )

    with mlflow.start_run(experiment_id=experimentID, run_name=run_name) as run:
        rf = RandomForestClassifier(**params)
        rf.fit(X_train, y_train)
        predictions = rf.predict(X_test)
        probabilities = rf.predict_proba(X_test)[:, 1]

        mlflow.sklearn.log_model(rf, "random-forest-model")

        [mlflow.log_param(param, value) for param, value in params.items()]
        mlflow.log_param("dataset_path", DATASET_PATH)
        mlflow.log_param("target_column", "podium_finish")
        mlflow.log_param("feature_cols", ", ".join(feature_cols))

        accuracy = accuracy_score(y_test, predictions)
        precision = precision_score(y_test, predictions, zero_division=0)
        recall = recall_score(y_test, predictions, zero_division=0)
        f1 = f1_score(y_test, predictions, zero_division=0)
        balanced_accuracy = balanced_accuracy_score(y_test, predictions)
        roc_auc = roc_auc_score(y_test, probabilities)

        print("accuracy:", accuracy)
        print("precision:", precision)
        print("recall:", recall)
        print("f1:", f1)
        print("balanced_accuracy:", balanced_accuracy)
        print("roc_auc:", roc_auc)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1", f1)
        mlflow.log_metric("balanced_accuracy", balanced_accuracy)
        mlflow.log_metric("roc_auc", roc_auc)

        importance = pd.DataFrame(
            list(zip(feature_cols, rf.feature_importances_)),
            columns=["Feature", "Importance"],
        ).sort_values("Importance", ascending=False)

        predictions_df = pd.DataFrame({
            "actual": y_test,
            "predicted": predictions,
            "prediction_probability": probabilities,
        })

        report_df = pd.DataFrame(
            classification_report(y_test, predictions, output_dict=True, zero_division=0)
        ).transpose()

        fig, ax = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay.from_predictions(y_test, predictions, ax=ax)
        plt.title("Confusion Matrix")

        with tempfile.TemporaryDirectory(prefix="f1-mlflow-artifacts-") as temp_dir:
            temp_dir_path = Path(temp_dir)
            importance.to_csv(temp_dir_path / "feature-importance.csv", index=False)
            predictions_df.to_csv(temp_dir_path / "predictions.csv", index=False)
            report_df.to_csv(temp_dir_path / "classification-report.csv")
            fig.savefig(temp_dir_path / "confusion-matrix.png", bbox_inches="tight")
            mlflow.log_artifacts(temp_dir, "artifacts")

        display(fig)
        plt.close(fig)

        return {
            "run_id": run.info.run_id,
            "run_name": run_name,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "balanced_accuracy": balanced_accuracy,
            "roc_auc": roc_auc,
        }


run with different parameters


In [0]:
run_results = []


In [0]:
params = {
    "n_estimators": 50,
    "max_depth": 5,
    "random_state": 42,
}

run_results.append(log_rf(experimentID, "Run 1: depth5 trees50", params, X_train, X_test, y_train, y_test, feature_cols))


In [0]:
params = {
    "n_estimators": 100,
    "max_depth": 5,
    "random_state": 42,
}

run_results.append(log_rf(experimentID, "Run 2: depth5 trees100", params, X_train, X_test, y_train, y_test, feature_cols))


In [0]:
params = {
    "n_estimators": 200,
    "max_depth": 5,
    "random_state": 42,
}

run_results.append(log_rf(experimentID, "Run 3: depth5 trees200", params, X_train, X_test, y_train, y_test, feature_cols))


In [0]:
params = {
    "n_estimators": 50,
    "max_depth": 10,
    "random_state": 42,
}

run_results.append(log_rf(experimentID, "Run 4: depth10 trees50", params, X_train, X_test, y_train, y_test, feature_cols))


In [0]:
params = {
    "n_estimators": 100,
    "max_depth": 10,
    "random_state": 42,
}

run_results.append(log_rf(experimentID, "Run 5: depth10 trees100", params, X_train, X_test, y_train, y_test, feature_cols))


In [0]:
params = {
    "n_estimators": 200,
    "max_depth": 10,
    "random_state": 42,
}

run_results.append(log_rf(experimentID, "Run 6: depth10 trees200", params, X_train, X_test, y_train, y_test, feature_cols))


In [0]:
params = {
    "n_estimators": 50,
    "max_depth": 20,
    "random_state": 42,
}

run_results.append(log_rf(experimentID, "Run 7: depth20 trees50", params, X_train, X_test, y_train, y_test, feature_cols))


In [0]:
params = {
    "n_estimators": 100,
    "max_depth": 20,
    "random_state": 42,
}

run_results.append(log_rf(experimentID, "Run 8: depth20 trees100", params, X_train, X_test, y_train, y_test, feature_cols))


In [0]:
params = {
    "n_estimators": 200,
    "max_depth": 20,
    "random_state": 42,
}

run_results.append(log_rf(experimentID, "Run 9: depth20 trees200", params, X_train, X_test, y_train, y_test, feature_cols))


In [0]:
params = {
    "n_estimators": 300,
    "max_depth": 20,
    "random_state": 42,
}

run_results.append(log_rf(experimentID, "Run 10: depth20 trees300", params, X_train, X_test, y_train, y_test, feature_cols))


In [0]:
leaderboard = pd.DataFrame(run_results).sort_values("f1", ascending=False).reset_index(drop=True)
display(leaderboard)


#### Best Model

I selected Run 5 as the best model because it had the highest F1 score (0.5000). Since podium prediction is an imbalanced binary classification problem, F1 is a better primary metric than accuracy alone. The model also achieved 0.8976 accuracy and 0.9058 ROC-AUC, and its MLflow artifacts helped confirm the run behaved reasonably.


In [0]:
best_run = leaderboard.iloc[0]
print("Best run:", best_run["run_name"])
print("Best run_id:", best_run["run_id"])
print("Best f1:", best_run["f1"])
